In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, log_loss

import tensorflow as tf
from tensorflow.keras import layers, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter


# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)


# ======================
# CNN FEATURE EXTRACTOR
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)

    feat = layers.Dense(512, activation="relu", name="feat")(x)

    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn = Model(inp, out)
    backbone = Model(inp, feat)

    return cnn, backbone


cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []

    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())

    return np.vstack(X), np.concatenate(Y)


X_all, y_all = extract_features(all_ds)

print("CNN feature shape:", X_all.shape)


# ======================
# PCA + STANDARDIZATION
# ======================
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

F = X_tr.shape[1]

X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

print("Final feature shape for graph:", X_std.shape)


# ======================
# kNN GRAPH BUILDING
# ======================
nbrs = NearestNeighbors(
    n_neighbors=K_DEFAULT + 1,
    metric="cosine"
).fit(X_std)

dist, idx_knn = nbrs.kneighbors(X_std)

rows, cols, data = [], [], []

for i in range(N):
    for j, d in zip(idx_knn[i], dist[i]):
        if i == j:
            continue

        sim = 1.0 - float(d)

        if sim <= 0:
            continue

        rows.append(i)
        cols.append(j)
        data.append(sim)

A = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
A = (A + A.T).tocsr()
A.setdiag(1.0)

A_norm = gcn_filter(A)

nnz_total = A.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2
degrees = np.array(A.sum(axis=1)).flatten() - 1

n_comp, labels_comp = connected_components(A, directed=False)

print("Graph edges:", undirected_edges)
print("Graph components:", n_comp)


# ======================
# GCN CLASSIFIER
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )

    return model


Y = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(X_std.shape[1])


# ======================
# TRAIN GCN
# ======================
history = gcn.fit(
    [X_std, A_norm],
    Y,
    sample_weight=mask_tr.astype(np.float32),
    validation_data=(
        [X_std, A_norm],
        Y,
        mask_va.astype(np.float32)
    ),
    epochs=EPOCHS_GCN,
    batch_size=N,
    shuffle=False,
    verbose=1,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=30,
            restore_best_weights=True
        )
    ]
)


# ======================
# FINAL PREDICTION
# ======================
pred_prob = gcn.predict(
    [X_std, A_norm],
    batch_size=N,
    verbose=0
)

pred = np.argmax(pred_prob, axis=1)

print("Validation Accuracy:", accuracy_score(labels[mask_va], pred[mask_va]))
print("Test Accuracy:", accuracy_score(labels[mask_te], pred[mask_te]))

In [ ]:
print("Test Accuracy:", accuracy_score(labels[mask_te], pred[mask_te]))

In [ ]:
# Save the entire GCN model
gcn.save("Tumor.h5")

print("Saved full model to disk.")

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from lime.lime_image import LimeImageExplainer
from skimage.segmentation import mark_boundaries

classes = {i: class_names[i] for i in range(len(class_names))}
explainer = LimeImageExplainer()

def load_image_numpy(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img.numpy()

def select_two_test_images_per_class(labels, idx_te, classes):
    selected_idx = []
    selected_labels = []

    for class_id in classes.keys():
        class_test_idx = idx_te[labels[idx_te] == class_id]
        take_n = min(2, len(class_test_idx))

        for j in range(take_n):
            selected_idx.append(class_test_idx[j])
            selected_labels.append(class_id)

    return np.array(selected_idx), np.array(selected_labels)

selected_node_idx, y_selected = select_two_test_images_per_class(labels, idx_te, classes)
X_selected = np.array([load_image_numpy(paths[i]) for i in selected_node_idx])

def preprocess_for_gcn(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)

    if tf.reduce_max(images) > 1.0:
        images = images / 255.0

    images = tf.image.resize(images, IMG_SIZE)

    feat = backbone(images, training=False).numpy()

    if USE_PCA:
        feat = pca.transform(feat)

    feat = scaler.transform(feat)

    return feat.astype(np.float32)

def make_predict_fn(node_index):
    def predict_fn(images):
        feats = preprocess_for_gcn(images)
        outputs = []

        for feat in feats:
            X_temp = X_std.copy()
            X_temp[node_index] = feat

            prob = gcn.predict(
                [X_temp, A_norm],
                batch_size=N,
                verbose=0
            )

            outputs.append(prob[node_index])

        return np.array(outputs)

    return predict_fn

def generate_lime_explanations_gcn(X_samples, y_samples, node_indices, classes, num_samples=100, num_features=5):
    for i, image in enumerate(X_samples):
        true_class = y_samples[i]
        node_index = node_indices[i]
        predict_fn = make_predict_fn(node_index)

        predicted_probs = predict_fn(np.array([image]))[0]
        predicted_class = np.argmax(predicted_probs)

        print(f"\nGenerating explanation for class: {classes[true_class]}")
        print("Model Prediction Probabilities:")

        for idx, prob in enumerate(predicted_probs):
            print(f" - {classes[idx]}: {prob:.4f}")

        print(f"Predicted Class: {classes[predicted_class]} (Probability: {predicted_probs[predicted_class]:.4f})")
        print("\nExplanation of Red and Green Marks in LIME Visualization:")
        print(f" - Green regions: Areas that positively contributed to the predicted class ({classes[predicted_class]})")
        print(" - Red regions: Areas that negatively contributed to the predicted class.")
        print(" - Intensity reflects the magnitude of the contribution.")

        explanation = explainer.explain_instance(
            image=image.astype(np.double),
            classifier_fn=predict_fn,
            top_labels=2,
            hide_color=0,
            num_samples=num_samples
        )

        temp, mask = explanation.get_image_and_mask(
            label=predicted_class,
            positive_only=False,
            num_features=num_features,
            hide_rest=False
        )

        temp_show = temp.copy()

        if temp_show.max() <= 1.0:
            temp_show = temp_show * 255.0

        colored_mask = np.zeros_like(temp_show)
        colored_mask[mask == 1] = [0, 255, 0]
        colored_mask[mask == -1] = [255, 0, 0]

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        axs[0].imshow(np.clip(image, 0, 255).astype(np.uint8))
        axs[0].set_title(f"Input: {classes[true_class]}")
        axs[0].axis("off")

        axs[1].imshow(np.zeros_like(temp_show).astype(np.uint8))
        axs[1].imshow(np.clip(colored_mask, 0, 255).astype(np.uint8), alpha=0.6)
        axs[1].set_title("Features: Green Positive, Red Negative")
        axs[1].axis("off")

        axs[2].imshow(np.clip(image, 0, 255).astype(np.uint8))
        axs[2].imshow(mark_boundaries(temp_show.astype(np.uint8) / 255.0, mask), alpha=0.6)
        axs[2].set_title(f"Output: {classes[predicted_class]}")
        axs[2].axis("off")

        plt.tight_layout()
        plt.show()

generate_lime_explanations_gcn(
    X_samples=X_selected,
    y_samples=y_selected,
    node_indices=selected_node_idx,
    classes=classes,
    num_samples=100,
    num_features=5
)

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from lime.lime_image import LimeImageExplainer
from skimage.segmentation import mark_boundaries

classes = {i: class_names[i] for i in range(len(class_names))}
explainer = LimeImageExplainer()

def load_image_numpy(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img.numpy()

def select_two_test_images_per_class(labels, idx_te, classes):
    selected_idx = []
    selected_labels = []

    for class_id in classes.keys():
        class_indices = idx_te[labels[idx_te] == class_id]
        selected_idx.append(class_indices[0])
        selected_idx.append(class_indices[1])
        selected_labels.append(class_id)
        selected_labels.append(class_id)

    return np.array(selected_idx), np.array(selected_labels)

selected_node_idx, y_selected = select_two_test_images_per_class(labels, idx_te, classes)
X_selected = np.array([load_image_numpy(paths[i]) for i in selected_node_idx])

def preprocess_for_gcn(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)

    if tf.reduce_max(images) > 1.0:
        images = images / 255.0

    images = tf.image.resize(images, IMG_SIZE)
    feat = backbone(images, training=False).numpy()

    if USE_PCA:
        feat = pca.transform(feat)

    feat = scaler.transform(feat)
    return feat.astype(np.float32)

def make_predict_fn(node_index):
    def predict_fn(images):
        feats = preprocess_for_gcn(images)
        outputs = []

        for feat in feats:
            X_temp = X_std.copy()
            X_temp[node_index] = feat

            prob = gcn.predict(
                [X_temp, A_norm],
                batch_size=N,
                verbose=0
            )

            outputs.append(prob[node_index])

        return np.array(outputs)

    return predict_fn

def generate_lime_explanations_gcn(X_samples, y_samples, node_indices, classes, num_samples=100, num_features=5):
    for i, image in enumerate(X_samples):
        class_id = y_samples[i]
        node_index = node_indices[i]
        predict_fn = make_predict_fn(node_index)

        predicted_class_probabilities = predict_fn(np.array([image]))[0]
        predicted_class = np.argmax(predicted_class_probabilities)

        print(f"\nGenerating explanation for class: {classes[class_id]}")
        print("Model Prediction Probabilities:")

        for idx, prob in enumerate(predicted_class_probabilities):
            print(f" - {classes[idx]}: {prob:.4f}")

        print(f"Predicted Class: {classes[predicted_class]} (Probability: {predicted_class_probabilities[predicted_class]:.4f})")

        print("\nExplanation of Red and Green Marks in LIME Visualization:")
        print(f" - Green regions: Areas that positively contributed to the predicted class ({classes[predicted_class]})")
        print(" - Red regions: Areas that negatively contributed to the predicted class.")
        print(" - Intensity reflects the magnitude of the contribution.")

        explanation = explainer.explain_instance(
            image=image.astype(np.double),
            classifier_fn=predict_fn,
            top_labels=2,
            hide_color=0,
            num_samples=num_samples
        )

        temp, mask = explanation.get_image_and_mask(
            label=predicted_class,
            positive_only=False,
            num_features=num_features,
            hide_rest=False
        )

        temp = temp[:image.shape[0], :image.shape[1]]
        mask = mask[:image.shape[0], :image.shape[1]]

        if temp.max() <= 1.0:
            temp = temp * 255.0

        colored_mask = np.zeros_like(temp)
        colored_mask[mask == 1] = [0, 255, 0]
        colored_mask[mask == -1] = [255, 0, 0]

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        axs[0].imshow(np.clip(image, 0, 255).astype(np.uint8))
        axs[0].set_title(f"Input: {classes[class_id]}")
        axs[0].axis("off")

        axs[1].imshow(np.zeros_like(temp).astype(np.uint8))
        axs[1].imshow(np.clip(colored_mask, 0, 255).astype(np.uint8), alpha=0.6)
        axs[1].set_title("Features: Green Positive, Red Negative")
        axs[1].axis("off")

        axs[2].imshow(np.clip(image, 0, 255).astype(np.uint8))
        axs[2].imshow(mark_boundaries(temp.astype(np.uint8) / 255.0, mask), alpha=0.6)
        axs[2].set_title(f"Output: {classes[predicted_class]}")
        axs[2].axis("off")

        plt.tight_layout()
        plt.show()

generate_lime_explanations_gcn(
    X_samples=X_selected,
    y_samples=y_selected,
    node_indices=selected_node_idx,
    classes=classes,
    num_samples=100,
    num_features=5
)

In [ ]:
import os
import cv2
import numpy as np
import shap
import matplotlib.pyplot as plt
import tensorflow as tf
import logging

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel(logging.ERROR)

classes = {i: class_names[i] for i in range(len(class_names))}

def load_image_numpy(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img.numpy()

def select_one_test_image_per_class(labels, idx_te, classes):
    selected_idx = []
    selected_labels = []

    for class_id in classes.keys():
        class_indices = idx_te[labels[idx_te] == class_id]
        selected_idx.append(class_indices[0])
        selected_labels.append(class_id)

    return np.array(selected_idx), np.array(selected_labels)

selected_node_idx, y_selected = select_one_test_image_per_class(labels, idx_te, classes)
X_selected = np.array([load_image_numpy(paths[i]) for i in selected_node_idx])
X_selected_norm = X_selected / 255.0

def preprocess_for_gcn(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)

    if tf.reduce_max(images) > 1.0:
        images = images / 255.0

    images = tf.image.resize(images, IMG_SIZE)
    feat = backbone(images, training=False).numpy()

    if USE_PCA:
        feat = pca.transform(feat)

    feat = scaler.transform(feat)
    return feat.astype(np.float32)

def make_shap_predict_fn(node_index):
    def predict_fn(images):
        feats = preprocess_for_gcn(images)
        outputs = []

        for feat in feats:
            X_temp = X_std.copy()
            X_temp[node_index] = feat

            prob = gcn.predict(
                [X_temp, A_norm],
                batch_size=N,
                verbose=0
            )

            outputs.append(prob[node_index])

        return np.array(outputs)

    return predict_fn

def generate_shap_explanations_gcn(X_test, y_test, node_indices, classes, max_evals=500, batch_size=50):
    for index, image in enumerate(X_test):
        true_class = y_test[index]
        node_index = node_indices[index]
        predict_fn = make_shap_predict_fn(node_index)

        print(f"\nGenerating SHAP explanation for class: {classes[true_class]}")

        masker = shap.maskers.Image("blur(16,16)", image.shape)

        explainer = shap.Explainer(
            predict_fn,
            masker,
            output_names=[classes[i] for i in range(len(classes))]
        )

        shap_values = explainer(
            image[np.newaxis, ...],
            max_evals=max_evals,
            batch_size=batch_size
        )

        predictions = predict_fn(image[np.newaxis, ...])[0]
        predicted_class = np.argmax(predictions)

        print("\nModel Prediction Probabilities:")
        for i, prob in enumerate(predictions):
            print(f" - {classes[i]}: {prob:.4f}")

        print(f"Predicted Class: {classes[predicted_class]} (Probability: {predictions[predicted_class]:.4f})")

        values_all = shap_values.values

        img_plot = np.squeeze(image)

        if img_plot.max() <= 1.0:
            img_plot = img_plot * 255.0

        img_plot = np.clip(img_plot, 0, 255).astype(np.uint8)

        fig = plt.figure(figsize=(18, 8))

        ax_img = plt.subplot2grid(
            (2, len(classes) + 2),
            (0, 0),
            rowspan=1,
            colspan=2
        )

        ax_img.imshow(img_plot)
        ax_img.axis("off")
        ax_img.set_title("Original Image", fontsize=14, fontweight="bold")

        ax_prob = plt.subplot2grid(
            (2, len(classes) + 2),
            (0, 2),
            rowspan=1,
            colspan=len(classes)
        )

        ax_prob.axis("off")

        ax_prob.text(
            0.0,
            0.95,
            f"Predicted Class: {classes[predicted_class]}",
            fontsize=16,
            fontweight="bold",
            transform=ax_prob.transAxes
        )

        y_pos = 0.72

        for c in range(len(classes)):
            color = "red" if c == predicted_class else "black"
            weight = "bold" if c == predicted_class else "normal"

            ax_prob.text(
                0.02,
                y_pos,
                f"{classes[c]}:",
                fontsize=13,
                color=color,
                fontweight=weight,
                transform=ax_prob.transAxes
            )

            ax_prob.text(
                0.35,
                y_pos,
                f"{predictions[c]:.4f}",
                fontsize=13,
                color=color,
                fontweight=weight,
                transform=ax_prob.transAxes
            )

            y_pos -= 0.17

        heat_axes = []

        for c in range(len(classes)):
            if values_all.ndim == 5:
                vals_c = values_all[..., c]
            else:
                vals_c = values_all

            vals_c = np.squeeze(vals_c)

            heat = np.mean(vals_c, axis=-1)
            vmax = np.max(np.abs(heat)) + 1e-8

            ax = plt.subplot2grid(
                (2, len(classes) + 2),
                (1, c),
                rowspan=1,
                colspan=1
            )

            ax.imshow(img_plot, alpha=0.35)

            im = ax.imshow(
                heat,
                cmap="bwr",
                alpha=0.75,
                vmin=-vmax,
                vmax=vmax
            )

            ax.axis("off")
            ax.set_title(classes[c], fontsize=11, fontweight="bold")

            heat_axes.append(ax)

        ax_last = plt.subplot2grid(
            (2, len(classes) + 2),
            (1, len(classes)),
            rowspan=1,
            colspan=2
        )

        if values_all.ndim == 5:
            vals_pred = values_all[..., predicted_class]
        else:
            vals_pred = values_all

        vals_pred = np.squeeze(vals_pred)

        heat_pred = np.mean(vals_pred, axis=-1)
        vmax_pred = np.max(np.abs(heat_pred)) + 1e-8

        ax_last.imshow(img_plot, alpha=0.35)

        im_last = ax_last.imshow(
            heat_pred,
            cmap="bwr",
            alpha=0.75,
            vmin=-vmax_pred,
            vmax=vmax_pred
        )

        ax_last.axis("off")
        ax_last.set_title(
            f"SHAP for {classes[predicted_class]}",
            fontsize=11,
            fontweight="bold"
        )
        plt.subplots_adjust(bottom=0.18)

        cbar_ax = fig.add_axes([0.08, 0.08, 0.84, 0.035])

        cbar = fig.colorbar(
            im_last,
            cax=cbar_ax,
            orientation="horizontal"
        )

        cbar.set_label("SHAP value")

        fig.suptitle(
            f"SHAP Explanation for Class: {classes[true_class]}",
            fontsize=16,
            fontweight="bold"
        )

        plt.tight_layout(rect=[0, 0.15, 1, 0.95])
        plt.show()
        plt.clf()

generate_shap_explanations_gcn(
    X_test=X_selected_norm,
    y_test=y_selected,
    node_indices=selected_node_idx,
    classes=classes,
    max_evals=500,
    batch_size=50
)

In [ ]:
import os
import numpy as np
import shap
import matplotlib.pyplot as plt
import tensorflow as tf
import logging

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel(logging.ERROR)

classes = {i: class_names[i] for i in range(len(class_names))}

def load_image_numpy(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img.numpy()

def select_one_test_image_per_class(labels, idx_te, classes):
    selected_idx = []
    selected_labels = []

    for class_id in classes.keys():
        class_indices = idx_te[labels[idx_te] == class_id]
        selected_idx.append(class_indices[0])
        selected_labels.append(class_id)

    return np.array(selected_idx), np.array(selected_labels)

selected_node_idx, y_selected = select_one_test_image_per_class(labels, idx_te, classes)
X_selected = np.array([load_image_numpy(paths[i]) for i in selected_node_idx])
X_selected_norm = X_selected / 255.0

def preprocess_for_gcn(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)

    if tf.reduce_max(images) > 1.0:
        images = images / 255.0

    images = tf.image.resize(images, IMG_SIZE)
    feat = backbone(images, training=False).numpy()

    if USE_PCA:
        feat = pca.transform(feat)

    feat = scaler.transform(feat)
    return feat.astype(np.float32)

def make_shap_predict_fn(node_index):
    def predict_fn(images):
        feats = preprocess_for_gcn(images)
        outputs = []

        for feat in feats:
            X_temp = X_std.copy()
            X_temp[node_index] = feat

            prob = gcn.predict(
                [X_temp, A_norm],
                batch_size=N,
                verbose=0
            )

            outputs.append(prob[node_index])

        return np.array(outputs)

    return predict_fn

def generate_shap_explanations_gcn(X_test, y_test, node_indices, classes, max_evals=500, batch_size=50):
    for index, image in enumerate(X_test):
        true_class = y_test[index]
        node_index = node_indices[index]
        predict_fn = make_shap_predict_fn(node_index)

        print(f"\nGenerating SHAP explanation for class: {classes[true_class]}")

        masker = shap.maskers.Image("blur(16,16)", image.shape)

        explainer = shap.Explainer(
            predict_fn,
            masker,
            output_names=[classes[i] for i in range(len(classes))]
        )

        shap_values = explainer(
            image[np.newaxis, ...],
            max_evals=max_evals,
            batch_size=batch_size
        )

        predictions = predict_fn(image[np.newaxis, ...])[0]
        predicted_class = np.argmax(predictions)

    
        print("\nModel Prediction Probabilities:")

        plt.figure(figsize=(8, 4))

        class_labels = [classes[i] for i in range(len(classes))]
        class_probs = [float(predictions[i]) for i in range(len(classes))]

        bars = plt.bar(class_labels, class_probs)

        plt.ylim(0, 1)
        plt.ylabel("Probability")
        plt.title("Class Prediction Probabilities")

        for bar, prob in zip(bars, class_probs):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{prob:.4f}",
                ha='center'
            )

        plt.show()




        print(f"Predicted Class: {classes[predicted_class]} (Probability: {predictions[predicted_class]:.4f})")

        print("\nExplanation of Red and Blue Marks in SHAP Visualization:")
        print(" - Red regions: Areas that positively contributed to the predicted class.")
        print(" - Blue regions: Areas that negatively contributed to the predicted class.")
        print(" - Intensity reflects the magnitude of the contribution.\n")

        mean_shap_value = np.mean(np.abs(shap_values.values))
        print("SHAP Values Summary:")
        print(f" - Mean SHAP value absolute: {mean_shap_value:.4f}")

        values = shap_values.values

        if values.ndim == 5:
            values = values[..., predicted_class]

        shap_values_fixed = shap.Explanation(
            values=values,
            base_values=None,
            data=image[np.newaxis, ...],
            output_names=[classes[predicted_class]]
        )

        img_plot = np.squeeze(image)

        if img_plot.max() <= 1.0:
            img_plot = img_plot * 255.0

        img_plot = np.clip(img_plot, 0, 255).astype(np.uint8)

        val_plot = values

        if val_plot.ndim == 5:
            val_plot = val_plot[..., predicted_class]

        val_plot = np.squeeze(val_plot)

        heat = np.mean(np.abs(val_plot), axis=-1)
        heat = heat / (heat.max() + 1e-8)

        heat_u8 = np.uint8(255 * heat)

        heatmap = cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        overlay = cv2.addWeighted(
            img_plot,
            0.6,
            heatmap,
            0.4,
            0
        )

        prob_text = "\n".join([
            f"{classes[i]}: {predictions[i]:.4f}"
            for i in range(len(classes))
        ])

        title_text = (
            f"Predicted: {classes[predicted_class]}\n\n"
            f"{prob_text}"
        )

        plt.figure(figsize=(8, 8))

        plt.imshow(overlay)
        plt.title(title_text, fontsize=10)
        plt.axis("off")

        plt.show()
        plt.clf()


generate_shap_explanations_gcn(
    X_test=X_selected_norm,
    y_test=y_selected,
    node_indices=selected_node_idx,
    classes=classes,
    max_evals=500,
    batch_size=50
)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

A_coo = A_norm.tocoo()

A_tf = tf.sparse.SparseTensor(
    indices=np.array([A_coo.row, A_coo.col]).T,
    values=A_coo.data.astype(np.float32),
    dense_shape=A_coo.shape
)

A_tf = tf.sparse.reorder(A_tf)

def make_gradcam_heatmap_gcn(img_array, node_index, last_conv_layer_name, pred_index=None):
    conv_layer = backbone.get_layer(last_conv_layer_name)

    grad_model = tf.keras.models.Model(
        inputs=backbone.input,
        outputs=[conv_layer.output, backbone.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, features = grad_model(img_array, training=False)

        feat = features

        if USE_PCA:
            pca_components = tf.constant(pca.components_.T, dtype=tf.float32)
            pca_mean = tf.constant(pca.mean_, dtype=tf.float32)
            feat = tf.matmul(feat - pca_mean, pca_components)

        scaler_mean = tf.constant(scaler.mean_, dtype=tf.float32)
        scaler_scale = tf.constant(scaler.scale_, dtype=tf.float32)
        feat = (feat - scaler_mean) / scaler_scale

        X_temp_tf = tf.convert_to_tensor(X_std, dtype=tf.float32)

        X_temp_tf = tf.tensor_scatter_nd_update(
            X_temp_tf,
            indices=[[node_index]],
            updates=[feat[0]]
        )

        predictions = gcn([X_temp_tf, A_tf], training=False)

        if pred_index is None:
            pred_index = tf.argmax(predictions[node_index])

        loss = predictions[node_index, pred_index]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()
    heatmap = cv2.resize(heatmap, IMG_SIZE)

    return heatmap, predictions[node_index].numpy()

def generate_gradcam_gcn(X_images, node_indices, y_true, class_names, last_conv_layer_name, num_images=5):
    num_images = min(num_images, len(X_images))

    plt.figure(figsize=(15, num_images * 5))

    for i in range(num_images):
        img = X_images[i]

        img_array = np.expand_dims(img, axis=0).astype("float32")

        if img_array.max() > 1.0:
            img_array = img_array / 255.0

        heatmap, probs = make_gradcam_heatmap_gcn(
            img_array=img_array,
            node_index=node_indices[i],
            last_conv_layer_name=last_conv_layer_name
        )

        pred_label = np.argmax(probs)

        img_show = img.copy()

        if img_show.max() <= 1.0:
            img_show = img_show * 255.0

        img_show = np.clip(img_show, 0, 255).astype(np.uint8)

        gray = cv2.cvtColor(img_show, cv2.COLOR_RGB2GRAY)
        brain_mask = (gray > 15).astype(np.float32)
        brain_mask = cv2.GaussianBlur(brain_mask, (9, 9), 0)

        heatmap = heatmap * brain_mask
        heatmap = heatmap / (np.max(heatmap) + 1e-8)

        heatmap_uint8 = np.uint8(255 * heatmap)
        heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

        superimposed_img = cv2.addWeighted(img_show, 0.45, heatmap_color, 0.55, 0)

        plt.subplot(num_images, 2, 2 * i + 1)
        plt.imshow(img_show)
        plt.title(f"Original Image\nTrue: {class_names[y_true[i]]}\nPredicted: {class_names[pred_label]}")
        plt.axis("off")

        plt.subplot(num_images, 2, 2 * i + 2)
        plt.imshow(superimposed_img)
        plt.title("Grad-CAM")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

last_conv_layer_name = "conv2d_2"

generate_gradcam_gcn(
    X_images=X_selected,
    node_indices=selected_node_idx,
    y_true=y_selected,
    class_names=class_names,
    last_conv_layer_name=last_conv_layer_name,
    num_images=5
)

In [ ]:
def overlay_heatmap(img, heatmap):
    img_show = img.copy()

    if img_show.max() <= 1.0:
        img_show = img_show * 255.0

    img_show = np.clip(img_show, 0, 255).astype(np.uint8)

    gray = cv2.cvtColor(img_show, cv2.COLOR_RGB2GRAY)
    brain_mask = (gray > 15).astype(np.float32)
    brain_mask = cv2.GaussianBlur(brain_mask, (9, 9), 0)

    heatmap = heatmap * brain_mask
    heatmap = heatmap / (np.max(heatmap) + 1e-8)

    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(img_show, 0.45, heatmap_color, 0.55, 0)
    return overlay


def plot_gcn_gradcam_multiple_layers(X_images, node_indices, y_true, class_names, last_conv_layers, num_images=5):
    num_images = min(num_images, len(X_images))
    num_columns = len(last_conv_layers) + 2

    fig, axes = plt.subplots(
        num_images,
        num_columns,
        figsize=(4 * num_columns, 4 * num_images)
    )

    if num_images == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_images):
        img = X_images[i]

        img_array = np.expand_dims(img, axis=0).astype("float32")

        if img_array.max() > 1.0:
            img_array = img_array / 255.0

        heatmaps = []
        probs_final = None

        for layer_name in last_conv_layers:
            heatmap, probs = make_gradcam_heatmap_gcn(
                img_array=img_array,
                node_index=node_indices[i],
                last_conv_layer_name=layer_name
            )

            heatmaps.append(heatmap)
            probs_final = probs

        pred_label = np.argmax(probs_final)
        true_label = y_true[i]
        acc = 100.0 if pred_label == true_label else 0.0

        img_show = img.copy()

        if img_show.max() <= 1.0:
            img_show = img_show * 255.0

        img_show = np.clip(img_show, 0, 255).astype(np.uint8)

        axes[i, 0].imshow(img_show)
        axes[i, 0].axis("off")
        axes[i, 0].set_title(f"Real: {true_label}\nPred: {pred_label}")

        for j, layer_name in enumerate(last_conv_layers):
            overlay = overlay_heatmap(img, heatmaps[j])

            axes[i, j + 1].imshow(overlay)
            axes[i, j + 1].axis("off")
            axes[i, j + 1].set_title(f"GCN + {layer_name}\nAcc: {acc:.2f}%")

        avg_heatmap = np.mean(np.array(heatmaps), axis=0)
        avg_overlay = overlay_heatmap(img, avg_heatmap)

        axes[i, -1].imshow(avg_overlay)
        axes[i, -1].axis("off")
        axes[i, -1].set_title(f"GCN Avg Grad-CAM\nAcc: {acc:.2f}%")

    plt.tight_layout()
    plt.show()


last_conv_layers = ["conv2d", "conv2d_1", "conv2d_2"]

plot_gcn_gradcam_multiple_layers(
    X_images=X_selected,
    node_indices=selected_node_idx,
    y_true=y_selected,
    class_names=class_names,
    last_conv_layers=last_conv_layers,
    num_images=5
)

In [ ]:
class_names = class_names

pred_prob = gcn.predict(
    [X_std, A_tf],
    batch_size=N,
    verbose=0
)

predicted_classes = np.argmax(pred_prob, axis=1)

actual_classes = labels

misclassified_indices = np.where(
    (predicted_classes != actual_classes) & mask_te
)[0]

correctly_classified_indices = np.where(
    (predicted_classes == actual_classes) & mask_te
)[0]

X_misclassified = np.array([
    load_image_numpy(paths[i]) for i in misclassified_indices
])

y_true_misclassified = actual_classes[misclassified_indices]

y_pred_misclassified = predicted_classes[misclassified_indices]

print(f"Total misclassified test images: {len(misclassified_indices)}")

last_conv_layers = ["conv2d", "conv2d_1", "conv2d_2"]

plot_gcn_gradcam_multiple_layers(
    X_images=X_misclassified,
    node_indices=misclassified_indices,
    y_true=y_true_misclassified,
    class_names=class_names,
    last_conv_layers=last_conv_layers,
    num_images=5
)

In [ ]:
X_correct = np.array([
    load_image_numpy(paths[i]) for i in correctly_classified_indices
])

y_true_correct = actual_classes[correctly_classified_indices]
y_pred_correct = predicted_classes[correctly_classified_indices]

print(f"Total correctly classified test images: {len(correctly_classified_indices)}")

last_conv_layers = ["conv2d", "conv2d_1", "conv2d_2"]

plot_gcn_gradcam_multiple_layers(
    X_images=X_correct,
    node_indices=correctly_classified_indices,
    y_true=y_true_correct,
    class_names=class_names,
    last_conv_layers=last_conv_layers,
    num_images=5
)

In [ ]:
def generate_gradient_input_explanations_gcn(X_images, node_indices, y_true_labels, predicted_labels, class_names, num_images=5):
    num_images_to_display = min(num_images, len(X_images))

    for i in range(num_images_to_display):
        img = X_images[i].astype("float32")

        if img.max() > 1.0:
            img = img / 255.0

        img_tensor = tf.convert_to_tensor(img)
        img_tensor = tf.expand_dims(img_tensor, axis=0)

        with tf.GradientTape() as tape:
            tape.watch(img_tensor)

            features = backbone(img_tensor, training=False)

            feat = features

            if USE_PCA:
                pca_components = tf.constant(pca.components_.T, dtype=tf.float32)
                pca_mean = tf.constant(pca.mean_, dtype=tf.float32)
                feat = tf.matmul(feat - pca_mean, pca_components)

            scaler_mean = tf.constant(scaler.mean_, dtype=tf.float32)
            scaler_scale = tf.constant(scaler.scale_, dtype=tf.float32)
            feat = (feat - scaler_mean) / scaler_scale

            X_temp_tf = tf.convert_to_tensor(X_std, dtype=tf.float32)
            X_temp_tf = tf.tensor_scatter_nd_update(
                X_temp_tf,
                indices=[[node_indices[i]]],
                updates=[feat[0]]
            )

            preds = gcn([X_temp_tf, A_tf], training=False)

            class_index = predicted_labels[i]
            loss = preds[node_indices[i], class_index]

        gradients = tape.gradient(loss, img_tensor)[0]

        grad_input = gradients * img_tensor[0]

        grad_input = grad_input.numpy()
        grad_input -= grad_input.mean()
        grad_input /= (grad_input.std() + 1e-8)
        grad_input *= 0.1
        grad_input += 0.5
        grad_input = np.clip(grad_input, 0, 1)
        grad_input = (grad_input * 255).astype(np.uint8)

        img_show = X_images[i].copy()

        if img_show.max() <= 1.0:
            img_show = img_show * 255.0

        img_show = np.clip(img_show, 0, 255).astype(np.uint8)

        plt.figure(figsize=(10, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(img_show)
        plt.title(
            f"Original Image\nTrue: {class_names[y_true_labels[i]]}\nPredicted: {class_names[predicted_labels[i]]}"
        )
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(grad_input)
        plt.title("GCN Gradient * Input")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
generate_gradient_input_explanations_gcn(
    X_images=X_misclassified,
    node_indices=misclassified_indices,
    y_true_labels=y_true_misclassified,
    predicted_labels=y_pred_misclassified,
    class_names=class_names,
    num_images=5
)

In [ ]:
generate_gradient_input_explanations_gcn(
    X_images=X_correct,
    node_indices=correctly_classified_indices,
    y_true_labels=y_true_correct,
    predicted_labels=y_pred_correct,
    class_names=class_names,
    num_images=5
)